# FP32 trace collection (main incl. PR #2, `nf4_fp32_from_fp16`)

**Current usage**: do not use Run all. For the rank-mismatch investigation, run setup sections 0.1–0.4, then only the new 0.4a diagnostic when ready. The diagnostic is disabled by default and has not been executed. Keep the original strict checks; full collection remains blocked pending review.

| Stage | Contents | When to run |
|---|---|---|
| 0 | Settings, mount Drive, install environment, prepare prompt files | Every new session / after a disconnect |
| 1 | Correctness checks (06, 09) + small collection + audit + package for Alex | Now |
| 2 | Full collection (chunked, resumable) + merge + audit + package | **Only after Alex confirms stage 1 is compatible** |

Rules (from Alex and PRECISION.md):
* If any check fails, **do not loosen thresholds or drop prompts**; keep the output and send it to Alex.
* Every run writes to a new directory; existing FP16 / FP32 pilot evidence is never modified.
* Never infer missing labels or probabilities. The collector stops on any mismatch; that is expected.

## 0.1 Settings (the only cell you need to edit)

In [15]:
# ======== edit here only ========
# Existing FP16 / FP32 pilot evidence lives in ECE285_Medusa/20261028_192748_UTC/; this notebook only reads it
RUN_NAME        = "fp32_v1"              # use a new name for every fresh attempt, e.g. fp32_v2
MAX_NEW_TOKENS  = 64                     # ⚠ confirm with Alex first! pilot used 64, skeleton default is 256; used by 06/09/01
SMALL_CALIB_N   = 5                      # small run: first 5 pilot prompts as calib
SMALL_EVAL_N    = 3                      # small run: first 3 MT-Bench questions as eval
CHUNK_SIZE      = 25                     # prompts per chunk in the full run (a disconnect loses at most one chunk)
# ================================

PRECISION_MODE  = "nf4_fp32_from_fp16"
PROJECT_REPO    = "https://github.com/alextung2/ECE_285_Final_Project"
PROJECT_BRANCH  = "main"                                     # PR #2 merged into main on 2026-11-02
MEDUSA_REPO     = "https://github.com/FasterDecoding/Medusa.git"
MEDUSA_COMMIT   = "e2a5d20c048a9b0a4092e6933c34313687422518"  # revision cited in PRECISION.md
MTBENCH_URL     = "https://raw.githubusercontent.com/lm-sys/FastChat/v0.2.36/fastchat/llm_judge/data/mt_bench/question.jsonl"

from pathlib import Path
DRIVE_BASE = Path("/content/drive/MyDrive/ECE285_Medusa")
RUN        = DRIVE_BASE / RUN_NAME          # all outputs of this run go here
CODE       = Path("/content/ECE_285_Final_Project")
SKEL       = CODE / "medusa-proj-skeleton"
VENV_PY    = "/content/medusa-env/bin/python"
print("Output directory:", RUN)

Output directory: /content/drive/MyDrive/ECE285_Medusa/fp32_v1


## 0.2 Mount Drive and check the GPU

In [16]:
from google.colab import drive
drive.mount("/content/drive")
RUN.mkdir(parents=True, exist_ok=True)
(RUN / "logs").mkdir(exist_ok=True)
!nvidia-smi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mon Nov  9 07:10:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

## 0.3 Get the code (main, incl. PR #2) + pinned Medusa; verify the Drive originals

In [17]:
import subprocess, json, datetime
def sh(*cmd, cwd=None):
    out = subprocess.run(list(cmd), cwd=cwd, check=True, capture_output=True, text=True)
    return out.stdout.strip()

if not CODE.exists():
    sh("git", "clone", PROJECT_REPO, str(CODE))
sh("git", "fetch", "origin", PROJECT_BRANCH, cwd=CODE)
sh("git", "checkout", "-B", PROJECT_BRANCH, f"origin/{PROJECT_BRANCH}", cwd=CODE)
project_sha = sh("git", "rev-parse", "HEAD", cwd=CODE)

# Use the existing prompt / manifest originals on Drive (20261028_192748_UTC) instead of regenerating them,
# and check they are byte-identical to the versions on GitHub main
main_sha = project_sha
SRC = DRIVE_BASE / "20261028_192748_UTC"
for name in ["data_manifest.json", "train_prompts.json", "validation_prompts.json", "pilot_prompts.json"]:
    on_main = subprocess.run(["git", "show", f"origin/main:medusa_trace_collection/prompts/{name}"], cwd=CODE, check=True, capture_output=True).stdout
    assert (SRC / name).read_bytes() == on_main, f"{name} on Drive differs from GitHub main; stop and investigate"
print("Drive originals match GitHub main:", main_sha)

MEDUSA_DIR = Path("/content/Medusa")
if not MEDUSA_DIR.exists():
    sh("git", "clone", MEDUSA_REPO, str(MEDUSA_DIR))
sh("git", "checkout", MEDUSA_COMMIT, cwd=MEDUSA_DIR)
medusa_sha = sh("git", "rev-parse", "HEAD", cwd=MEDUSA_DIR)

print("project:", PROJECT_BRANCH, project_sha)
print("medusa :", medusa_sha)
assert (SKEL / "PRECISION.md").exists(), "Wrong branch: PRECISION.md not found"

Drive originals match GitHub main: cf8dba5afe975f62e9af52bd75404b5aaaf0f03b
project: main cf8dba5afe975f62e9af52bd75404b5aaaf0f03b
medusa : e2a5d20c048a9b0a4092e6933c34313687422518


## 0.5 Run helper (streams output and writes a log to Drive)

In [ ]:
import shlex, sys, time, os
def run(script, *args, log_name):
    """Run a skeleton script in medusa-env; stream output and save it to RUN/logs/. Raise and stop on failure."""
    log = RUN / "logs" / f"{log_name}_{time.strftime('%Y%m%d-%H%M%S')}.log"
    cmd = [VENV_PY, "-u", f"scripts/{script}", *args]
    print("$", " ".join(shlex.quote(c) for c in cmd), "\nlog:", log, flush=True)
    with open(log, "w") as f:
        f.write("$ " + " ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, cwd=SKEL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="", flush=True); f.write(line); f.flush()
        rc = p.wait()
        f.write(f"\n[exit code {rc}]\n")
    if rc != 0:
        raise RuntimeError(f"{script} failed (exit {rc}). Do not loosen thresholds or drop prompts; send the log {log} and the output directory to Alex.")
    return log

M = f"model.precision_mode={PRECISION_MODE}"

## 0.6 Prepare prompt files (keep original prompt_id values and check them against the manifest)

In [ ]:
import hashlib, urllib.request
P = RUN / "prompts"; P.mkdir(exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
load = lambda p: json.loads(Path(p).read_text(encoding="utf-8"))

manifest   = load(SRC / "data_manifest.json")
train      = load(SRC / "train_prompts.json")
validation = load(SRC / "validation_prompts.json")
pilot      = load(SRC / "pilot_prompts.json")

# Check: prompt-file IDs exactly match the manifest
assert [r["prompt_id"] for r in train] == manifest["train_prompt_ids"], "train prompts do not match the manifest"
assert [r["prompt_id"] for r in validation] == manifest["validation_prompt_ids"], "validation prompts do not match the manifest"
assert not set(manifest["train_prompt_ids"]) & set(manifest["validation_prompt_ids"]), "train/validation overlap"
assert [r["prompt_id"] for r in pilot] == manifest["pilot_prompt_ids"], "pilot prompts do not match the manifest"

# MT-Bench: 80 first-turn questions, prompt_id = question_id (81–160)
mt_raw = P / "mt_bench_question_fastchat_v0.2.36.jsonl"
if not mt_raw.exists():
    urllib.request.urlretrieve(MTBENCH_URL, mt_raw)
mt = [json.loads(l) for l in mt_raw.read_text(encoding="utf-8").splitlines() if l.strip()]
assert len(mt) == 80 and len({q["question_id"] for q in mt}) == 80
eval_full = [dict(prompt_id=q["question_id"], instruction=q["turns"][0]) for q in mt]

files = {
    "calib_full.json": train + validation,          # 250 prompts: 200 train + 50 validation
    "eval_full.json": eval_full,                    # 80 MT-Bench first turns
    "calib_small.json": pilot[:SMALL_CALIB_N],      # small run
    "eval_small.json": eval_full[:SMALL_EVAL_N],
}
for name, recs in files.items():
    dest = P / name
    text = json.dumps(recs, ensure_ascii=False, indent=1)
    if dest.exists():
        assert dest.read_text(encoding="utf-8") == text, f"{dest} already exists with different content; use a new RUN_NAME"
    else:
        dest.write_text(text, encoding="utf-8")

prompt_manifest = dict(
    calib=dict(dataset=manifest.get("dataset"), revision=manifest.get("revision"), seed=manifest.get("seed"),
               train_prompt_ids=manifest["train_prompt_ids"], validation_prompt_ids=manifest["validation_prompt_ids"]),
    eval=dict(dataset="mtbench", source=MTBENCH_URL, first_turn_only=True, prompt_ids=[r["prompt_id"] for r in eval_full]),
    sha256={str(p.relative_to(P)): sha(p) for p in sorted(P.rglob("*.json*")) if p.name != "prompt_manifest.json" and "chunks" not in p.parts},
)
(P / "prompt_manifest.json").write_text(json.dumps(prompt_manifest, indent=2))
for name, recs in files.items():
    print(f"{name:18s} {len(recs):4d} prompts  first IDs {[r['prompt_id'] for r in recs[:4]]}")

calib_full.json     250 prompts  first IDs [736, 375, 679, 588]
eval_full.json       80 prompts  first IDs [81, 82, 83, 84]
calib_small.json      5 prompts  first IDs [736, 375, 679, 588]
eval_small.json       3 prompts  first IDs [81, 82, 83]


---
# Stage 1: correctness checks + small collection (run now)

Each cell downloads/loads the model (the first download is ~14 GB). If any cell fails, **stop** and send `RUN/logs/` and the matching output directory to Alex.

### 1.1 Agreement with official Medusa (5 prompts, fixed 64 tokens)

In [ ]:
run("06_validate_official.py", "--set", M, f"out_dir={RUN}/stage1/official", log_name="06_official")

$ /content/medusa-env/bin/python -u scripts/06_validate_official.py --set model.precision_mode=nf4_fp32_from_fp16 out_dir=/content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/official 
log: /content/drive/MyDrive/ECE285_Medusa/fp32_v1/logs/06_official_20261109-050000.log
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _tor

PosixPath('/content/drive/MyDrive/ECE285_Medusa/fp32_v1/logs/06_official_20261109-050000.log')

### 1.2 Precision gate: known cases (incl. Chihuahua / lost-key), static tree + 5 dynamic budgets

A failed gate is **recorded, not hidden**: the cell prints which prompt / policy / round failed, writes
`stage1/gate_status_known_cases.json`, and lets the notebook continue so the rest of the stage-1 evidence
can be collected for Alex. Thresholds and prompts are unchanged. If the result file already exists the
check is not rerun (09 refuses to overwrite evidence).

In [ ]:
def gate_summary(report_path, name):
    """Summarize a 09 precision_validation.json and record the gate status next to it."""
    report = json.loads(Path(report_path).read_text())
    failures = []
    for i, p in enumerate(report["prompts"]):
        if "error" in p:
            failures.append(dict(prompt=i, text=p["prompt"], problem="error", detail=p["error"]))
        if not p.get("cached_uncached_agree", False):
            failures.append(dict(prompt=i, text=p["prompt"], problem="cached != uncached greedy"))
        for pol in p.get("policies", []):
            bad = [r for r in pol["rounds"]
                   if not (r["same_prefix"] and r["same_ranks"] and not r["label_mismatches"])]
            if not pol["exact_agreement"] or bad:
                failures.append(dict(prompt=i, text=p["prompt"], policy=pol["name"],
                                     exact_agreement=pol["exact_agreement"], bad_rounds=bad))
    status = dict(check=name, passed=report["passed"], report=str(report_path),
                  n_prompts=len(report["prompts"]), failures=failures)
    (RUN / "stage1" / f"gate_status_{name}.json").write_text(json.dumps(status, indent=2))
    print(f"[{name}] passed={report['passed']}  prompts={len(report['prompts'])}  failing items={len(failures)}")
    for f in failures:
        print("  FAIL:", json.dumps(f)[:600])
    return status


def run_gate(name, *args):
    out = RUN / "stage1" / name
    report_path = out / "results" / "precision_validation.json"
    if report_path.exists():
        print(f"{report_path} already exists; not rerunning, summarizing the existing evidence.")
    else:
        try:
            run("09_validate_precision.py", *args, "--set", M, f"speed.max_new_tokens={MAX_NEW_TOKENS}",
                f"out_dir={out}", log_name=f"09_{name}")
        except RuntimeError as exc:
            print("\nGATE FAILED (recorded; continuing):", exc)
        if not report_path.exists():
            raise RuntimeError(f"09 crashed before writing {report_path}; see the log and send it to Alex.")
    return gate_summary(report_path, name)


gate_known = run_gate("known_cases")

/content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/known_cases/results/precision_validation.json already exists; not rerunning, summarizing the existing evidence.
[known_cases] passed=False  prompts=7  failing items=1
  FAIL: {"prompt": 6, "text": "What breed dog is smallest?", "policy": "dynamic_B8", "exact_agreement": true, "bad_rounds": [{"position": 37, "same_prefix": true, "same_ranks": false, "max_probability_error": 4.172325134277344e-06, "label_mismatches": []}]}


### 1.2b Diagnose a rank mismatch (only if 1.2 / 1.3 reports `same_ranks: false`)

Reruns **one** prompt with **one** dynamic budget and, at every round where the online head top-10 differs
from the teacher-forced trace, saves per-head online tokens/logits/probabilities, the trace tokens/probabilities,
the adjacent logit gaps at the differing ranks, and whether the selected tree used a differing (head, rank).
Correctness only; nothing is changed in the gate. Defaults below are the 1.2 failure (prompt 6, B8).

In [ ]:
import os
DIAG_PROMPT, DIAG_INDEX, DIAG_BUDGET = "What breed dog is smallest?", 6, 8

diag_src = r"""
# Diagnose offline/online top-k rank mismatch for one prompt and one dynamic budget (correctness only).
import json
from pathlib import Path
from _common import parse


def extra(ap):
    ap.add_argument("--prompt", required=True)
    ap.add_argument("--prompt-index", type=int, required=True)
    ap.add_argument("--budget", type=int, required=True)
    ap.add_argument("--out", type=Path, required=True)


def main():
    args, cfg = parse("Diagnose offline/online rank mismatch", extra)
    import torch
    from mcal.collect import traces_for_prompt
    from mcal.data import vicuna_prompt
    from mcal.decode import plain_greedy_generate, medusa_greedy_generate, DynamicTree
    from mcal.estimators import MedusaPlusProb
    from mcal.model import load_medusa
    from mcal.validation import TraceObserver
    if args.out.exists():
        raise FileExistsError(f"{args.out} exists; keep previous evidence")
    model, tok = load_medusa(cfg)
    limit = cfg["speed"]["max_new_tokens"]
    ids = tok([vicuna_prompt(args.prompt)], return_tensors="pt").input_ids.to(model.device)
    reference, _ = plain_greedy_generate(model, ids, limit)
    ts, _ = traces_for_prompt(model, tok, args.prompt, args.prompt_index, max_new_tokens=limit, keep_tokens=True)
    details = []

    class Diag(TraceObserver):
        def __call__(self, ids_, choices, buffers, candidates, logits, head_logits):
            n = len(self.rows)
            super().__call__(ids_, choices, buffers, candidates, logits, head_logits)
            if len(self.rows) == n or self.rows[-1]["same_ranks"]:
                return
            pos, K = self.rows[-1]["position"], self.traces.topk
            hl = head_logits.float()
            vals, toks = hl.topk(K + 2, dim=-1)
            probs = torch.softmax(hl, -1)
            heads = []
            for h in range(hl.shape[0]):
                online, trace = toks[h, :K].tolist(), [int(t) for t in self.traces.topk_token[pos, h]]
                if online == trace:
                    continue
                diff = [k for k in range(K) if online[k] != trace[k]]
                gaps = {k: dict(above=(float(vals[h, k - 1] - vals[h, k]) if k > 0 else None),
                                below=float(vals[h, k] - vals[h, k + 1])) for k in diff}
                used = sorted({k for k in diff for path in choices if len(path) > h and path[h] == k})
                heads.append(dict(head=h, differing_ranks=diff, adjacent_logit_gaps=gaps,
                                  tree_uses_differing_ranks=used,
                                  online_tokens=toks[h].tolist(), online_logits=vals[h].tolist(),
                                  online_probs=probs[h].gather(-1, toks[h]).tolist(),
                                  trace_tokens=trace, trace_probs=[float(x) for x in self.traces.topk_prob[pos, h]],
                                  online_probs_of_trace_tokens=probs[h, torch.tensor(trace, device=hl.device)].tolist()))
            details.append(dict(position=pos, label_mismatches=self.rows[-1]["label_mismatches"],
                                max_probability_error=self.rows[-1]["max_probability_error"],
                                tree=[list(c) for c in choices], heads=heads))

    observer = Diag(ts, reference, ids.shape[1])
    output, _ = medusa_greedy_generate(model, ids, DynamicTree(MedusaPlusProb(), args.budget), limit, observer=observer)
    result = dict(prompt=args.prompt, prompt_index=args.prompt_index, budget=args.budget, max_new_tokens=limit,
                  execution_contract=model.execution_contract,
                  exact_agreement=bool(torch.equal(output, reference)),
                  offline_online_agree=observer.passed, n_rounds=len(observer.rows),
                  mismatch_positions=[d["position"] for d in details], mismatches=details)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(result, indent=2))
    print(json.dumps({k: v for k, v in result.items() if k not in ("mismatches", "execution_contract")}))
    for d in details:
        for h in d["heads"]:
            print(f"pos {d['position']} head {h['head']}: ranks {h['differing_ranks']} differ; "
                  f"adjacent logit gaps {h['adjacent_logit_gaps']}; used by tree: {h['tree_uses_differing_ranks']}")


if __name__ == "__main__":
    main()
"""
DIAG = RUN / "stage1" / "diagnostics"; DIAG.mkdir(parents=True, exist_ok=True)
diag_script = DIAG / "diag_rank_mismatch.py"
diag_script.write_text(diag_src)                     # saved next to its output so Alex sees exactly what ran
diag_out = DIAG / f"rank_mismatch_prompt{DIAG_INDEX}_B{DIAG_BUDGET}.json"
log = RUN / "logs" / f"diag_prompt{DIAG_INDEX}_B{DIAG_BUDGET}_{time.strftime('%Y%m%d-%H%M%S')}.log"
cmd = [VENV_PY, "-u", str(diag_script), "--prompt", DIAG_PROMPT, "--prompt-index", str(DIAG_INDEX),
       "--budget", str(DIAG_BUDGET), "--out", str(diag_out), "--set", M, f"speed.max_new_tokens={MAX_NEW_TOKENS}"]
env = dict(os.environ, PYTHONPATH=str(SKEL / "scripts"))
with open(log, "w") as f:
    p = subprocess.Popen(cmd, cwd=SKEL, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True); f.write(line)
    rc = p.wait(); f.write(f"\n[exit code {rc}]\n")
print("exit", rc, "| result:", diag_out, "| log:", log)

/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available` instead.
Using `is_flash_attn_available` is deprecated and will be removed in v4.38. Please use `is_flash_attn_2_available`

### 1.3 Precision gate: 20 pilot prompts (slow; a failure is recorded and the notebook continues)

In [ ]:
gate_pilot = run_gate("pilot_gate", "--prompt-file", str(SRC / "pilot_prompts.json"))

$ /content/medusa-env/bin/python -u scripts/09_validate_precision.py --prompt-file /content/drive/MyDrive/ECE285_Medusa/20261028_192748_UTC/pilot_prompts.json --set model.precision_mode=nf4_fp32_from_fp16 speed.max_new_tokens=64 out_dir=/content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/pilot_gate 
log: /content/drive/MyDrive/ECE285_Medusa/fp32_v1/logs/09_pilot_gate_20261109-051355.log
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch

### 1.4 Small collection (calib + eval, with `topk_token`)

In [ ]:
common = ["--set", M, "traces.keep_tokens=true", f"traces.max_new_tokens={MAX_NEW_TOKENS}", f"out_dir={RUN}/stage1/small"]
run("01_collect_traces.py", "--split", "calib", *common, f"traces.calib.prompt_file={P/'calib_small.json'}", log_name="01_small_calib")
run("01_collect_traces.py", "--split", "eval",  *common, f"traces.eval.prompt_file={P/'eval_small.json'}",  log_name="01_small_eval")

$ /content/medusa-env/bin/python -u scripts/01_collect_traces.py --split calib --set model.precision_mode=nf4_fp32_from_fp16 traces.keep_tokens=true traces.max_new_tokens=64 out_dir=/content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/small traces.calib.prompt_file=/content/drive/MyDrive/ECE285_Medusa/fp32_v1/prompts/calib_small.json 
log: /content/drive/MyDrive/ECE285_Medusa/fp32_v1/logs/01_small_calib_20261109-063003.log
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/content/medusa-env/lib/python3.11/site-packages/transformers/util

PosixPath('/content/drive/MyDrive/ECE285_Medusa/fp32_v1/logs/01_small_eval_20261109-063613.log')

### 1.5 Structural audit + quick look

In [ ]:
SMALL = RUN / "stage1" / "small" / "traces"
run("10_audit_traces.py", "--calib", str(SMALL / "calib.npz"), "--eval", str(SMALL / "eval.npz"),
    "--report", str(RUN / "stage1" / "small" / f"audit_{time.strftime('%Y%m%d-%H%M%S')}.json"), "--set", M,
    log_name="10_audit_small")

peek = r"""
import sys, json, numpy as np
for f in sys.argv[1:]:
    z = np.load(f, allow_pickle=False); meta = json.loads(str(z["meta"]))
    print(f); print("  arrays:", {k: z[k].shape for k in z.files if k != "meta"})
    print("  prompts:", sorted(set(z["prompt_id"].tolist())))
    print("  topk_token present:", "topk_token" in z.files, "| fraction gold_rank==-1:", float((z["gold_rank"] == -1).mean()))
    print("  meta keys:", sorted(meta))
"""
print(subprocess.run([VENV_PY, "-c", peek, str(SMALL / "calib.npz"), str(SMALL / "eval.npz")],
                     capture_output=True, text=True).stdout)

$ /content/medusa-env/bin/python -u scripts/10_audit_traces.py --calib /content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/small/traces/calib.npz --eval /content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/small/traces/eval.npz --report /content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/small/audit_20261109-064020.json --set model.precision_mode=nf4_fp32_from_fp16 
log: /content/drive/MyDrive/ECE285_Medusa/fp32_v1/logs/10_audit_small_20261109-064020.log
{
  "offline_checks_passed": true,
  "gpu_validation_performed": false,
  "label_truth_verified": false,
  "files": [
    {
      "path": "/content/drive/MyDrive/ECE285_Medusa/fp32_v1/stage1/small/traces/calib.npz",
      "sha256": "39ced50cac7a6e24aa992708504f2d5e606db443cbb6f1ee3d394ec4762ad034",
      "steps": 295,
      "prompts": 5,
      "n_heads": 5,
      "topk": 10,
      "source": "real",
      "execution_contract": {
        "schema_version": 1,
        "model": "FasterDecoding/medusa-vicuna-7b-v1.3",
        "precision_mode"

### 1.6 Package for Alex (zip saved on Drive)

In [ ]:
import shutil
zip_base = DRIVE_BASE / f"{RUN_NAME}_stage1_for_alex_{time.strftime('%Y%m%d-%H%M')}"
staging = Path("/content/_stage1_pkg"); shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(RUN / "stage1", staging / "stage1")
for d in ["logs", "environment", "prompts"]:
    shutil.copytree(RUN / d, staging / d)
print(shutil.make_archive(str(zip_base), "zip", staging))
print("Send this zip to Alex. Run stage 2 only after he confirms compatibility.")

/content/drive/MyDrive/ECE285_Medusa/fp32_v1_stage1_for_alex_20261109-0640.zip
Send this zip to Alex. Run stage 2 only after he confirms compatibility.


---
# Stage 2: full collection (only after Alex confirms)

* calib = 250 prompts (200 train + 50 validation), eval = 80 MT-Bench questions.
* Prompts are split into chunks of `CHUNK_SIZE`, one npz per chunk. **After a disconnect**: rerun 0.1–0.6, then 2.2; finished chunks are skipped automatically, and an unfinished checkpoint is moved to `partial_attempts/` (kept, never deleted).
* ⚠ Chunking + merging is added by this notebook; confirm with Alex before starting that he accepts it (both chunk files and merged files are delivered).
* If a prompt mismatches, the collector stops and writes `*.failure.json`. Do not rerun to skip it; send it to Alex.

In [ ]:
STAGE2_APPROVED = False   # set to True after Alex confirms stage 1
assert STAGE2_APPROVED, "Wait for Alex to confirm stage 1 compatibility before running stage 2"
FULL = RUN / "stage2" / "full"

AssertionError: Wait for Alex to confirm stage 1 compatibility before running stage 2

### 2.1 (Optional, only if Alex asks) precision gate on the 50 validation prompts

In [ ]:
RUN_VAL_GATE = False
if RUN_VAL_GATE:
    run("09_validate_precision.py", "--prompt-file", str(SRC / "validation_prompts.json"), "--set", M,
        f"speed.max_new_tokens={MAX_NEW_TOKENS}", f"out_dir={RUN}/stage2/validation_gate", log_name="09_validation_gate")

### 2.2 Chunked collection (safe to rerun; resumes automatically)

In [ ]:
def collect_chunks(split, prompt_file):
    recs = load(prompt_file)
    chunk_dir = P / "chunks"; chunk_dir.mkdir(exist_ok=True)
    tdir = FULL / "traces"; tdir.mkdir(parents=True, exist_ok=True)
    n = (len(recs) + CHUNK_SIZE - 1) // CHUNK_SIZE
    for i in range(n):
        tag = f"{split}_part{i:02d}"
        npz, done = tdir / f"{tag}.npz", tdir / f"{tag}.done.json"
        failure = tdir / f"{tag}.failure.json"
        if done.exists():
            print(f"[skip] {tag} already done"); continue
        if failure.exists():
            raise RuntimeError(f"{failure} exists: a prompt mismatched. Do not skip it; send it to Alex.")
        if npz.exists():   # checkpoint left by a disconnect: keep it, never delete
            keep = FULL / "partial_attempts"; keep.mkdir(exist_ok=True)
            npz.rename(keep / f"{tag}_{time.strftime('%Y%m%d-%H%M%S')}.npz")
            print(f"[kept] unfinished {tag} moved to partial_attempts/")
        chunk = recs[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE]
        cf = chunk_dir / f"{tag}.json"
        cf.write_text(json.dumps(chunk, ensure_ascii=False, indent=1), encoding="utf-8")
        t0 = time.time()
        log = run("01_collect_traces.py", "--split", split, "--set", M, "traces.keep_tokens=true",
                  f"traces.max_new_tokens={MAX_NEW_TOKENS}", f"traces.{split}.prompt_file={cf}",
                  f"traces.{split}.file=traces/{tag}.npz", f"out_dir={FULL}", log_name=f"01_{tag}")
        skipped = [l.strip() for l in log.read_text().splitlines() if l.startswith("skip prompt")]
        done.write_text(json.dumps(dict(chunk=tag, prompt_ids=[r["prompt_id"] for r in chunk],
                                        prompt_file_sha256=sha(cf), npz_sha256=sha(npz),
                                        skipped_lines=skipped, seconds=round(time.time() - t0, 1),
                                        log=str(log)), indent=2))
        print(f"[done] {tag}: {len(chunk)} prompts, {time.time() - t0:.0f}s, skipped {len(skipped)}")

collect_chunks("calib", P / "calib_full.json")
collect_chunks("eval",  P / "eval_full.json")

### 2.3 Merge chunks → `calib.npz` / `eval.npz` (chunk files are kept)

In [ ]:
merge = r"""
import sys, json, hashlib
from pathlib import Path
sys.path.insert(0, sys.argv[1])
from mcal.traces import TraceSet
full, split, prompt_file = Path(sys.argv[2]), sys.argv[3], Path(sys.argv[4])
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
dest = full / "traces" / f"{split}.npz"
if dest.exists():
    raise SystemExit(f"{dest} already exists; keeping it, not overwriting")
parts_f = sorted((full / "traces").glob(f"{split}_part*.npz"))
dones = [p.with_suffix(".done.json") for p in parts_f]
assert parts_f and all(d.exists() for d in dones), "some chunks are not finished yet"
parts = [TraceSet.load(p) for p in parts_f]
ts = TraceSet.concat(parts)          # checks that execution contracts match
recs = json.loads(prompt_file.read_text(encoding="utf-8"))
ids, prompts = [r["prompt_id"] for r in recs], [r["instruction"] for r in recs]
present = set(ts.prompt_id.tolist())
ts.meta.update(
    prompt_sha256=hashlib.sha256(json.dumps(list(zip(ids, prompts))).encode()).hexdigest(),  # same algorithm as 01, over all prompts
    greedy_agreement=None,
    merged_from=[dict(file=p.name, sha256=sha(p), n_steps=int(t.n_steps), prompt_sha256=t.meta.get("prompt_sha256"),
                      greedy_agreement=t.meta.get("greedy_agreement")) for p, t in zip(parts_f, parts)],
    expected_prompt_ids=ids,
    missing_prompt_ids=[i for i in ids if i not in present],
)
ts.save(dest)
print(split, ts.summary())
print("missing (skipped) prompts:", ts.meta["missing_prompt_ids"])
"""
for split, pf in [("calib", P / "calib_full.json"), ("eval", P / "eval_full.json")]:
    r = subprocess.run([VENV_PY, "-c", merge, str(SKEL / "src"), str(FULL), split, str(pf)], capture_output=True, text=True)
    print(r.stdout, r.stderr)

### 2.4 Audit + package

In [ ]:
run("10_audit_traces.py", "--calib", str(FULL / "traces" / "calib.npz"), "--eval", str(FULL / "traces" / "eval.npz"),
    "--report", str(FULL / f"audit_{time.strftime('%Y%m%d-%H%M%S')}.json"), "--set", M, log_name="10_audit_full")

zip_base = DRIVE_BASE / f"{RUN_NAME}_stage2_full_{time.strftime('%Y%m%d-%H%M')}"
staging = Path("/content/_stage2_pkg"); shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(RUN / "stage2", staging / "stage2")
for d in ["logs", "environment", "prompts"]:
    shutil.copytree(RUN / d, staging / d)
print(shutil.make_archive(str(zip_base), "zip", staging))
print("Deliver: npz files (merged + chunks), prompt_manifest / data_manifest / prompt files, environment, logs, audit and correctness reports.")